# Credit Card Fraud — Exploratory Data Analysis

Dataset: **Kaggle ULB Credit Card Fraud Detection** (`mlg-ulb/creditcardfraud`).

- 284,807 transactions over two days by European cardholders (Sept 2013)
- 492 frauds → **0.172%** positive rate (severe class imbalance)
- Features `V1`..`V28` are anonymized **PCA components**; `Time` and `Amount` are raw; `Class` is the label (1 = fraud)

This notebook covers: class balance, distribution plots, a correlation heatmap, and **statistical hypothesis testing** for which features separate fraud from legitimate transactions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
RANDOM_STATE = 42

In [ ]:
# Expects data/creditcard.csv relative to the repo root.
df = pd.read_csv('../data/creditcard.csv')
print(f'shape: {df.shape}')
df.head()

In [ ]:
# Basic integrity checks
print('missing values:', int(df.isna().sum().sum()))
print('duplicate rows :', int(df.duplicated().sum()))
df.describe().T[['mean', 'std', 'min', 'max']]

## 1. Class balance

The headline problem: fraud is rarer than 1 in 500.

In [ ]:
counts = df['Class'].value_counts().sort_index()
frac = counts / counts.sum()
summary = pd.DataFrame({'count': counts, 'pct': (frac * 100).round(3)})
summary.index = ['Legit (0)', 'Fraud (1)']
print(summary)
print(f"\nimbalance ratio: 1 fraud per {counts[0] / counts[1]:.0f} legit")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=summary.index, y='count', data=summary, ax=ax[0])
ax[0].set_title('Class counts (log scale)'); ax[0].set_yscale('log')
ax[1].pie(counts, labels=summary.index, autopct='%1.3f%%', startangle=90,
          colors=['#4c72b0', '#c44e52'])
ax[1].set_title('Class proportion')
plt.tight_layout(); plt.show()

## 2. Amount and Time distributions

How do fraudulent and legitimate transactions differ in monetary amount and timing?

In [ ]:
legit = df[df.Class == 0]
fraud = df[df.Class == 1]

print('Amount summary by class:')
print(df.groupby('Class')['Amount'].describe()[['mean', '50%', 'std', 'max']])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(np.log1p(legit.Amount), bins=60, alpha=0.6, label='legit', density=True)
ax[0].hist(np.log1p(fraud.Amount), bins=60, alpha=0.6, label='fraud', density=True)
ax[0].set_title('log1p(Amount) density'); ax[0].set_xlabel('log1p(Amount)'); ax[0].legend()

ax[1].hist(legit.Time / 3600, bins=48, alpha=0.6, label='legit', density=True)
ax[1].hist(fraud.Time / 3600, bins=48, alpha=0.6, label='fraud', density=True)
ax[1].set_title('Transaction time (hours) density'); ax[1].set_xlabel('hours since first txn'); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. Correlation heatmap

Because `V1`..`V28` are PCA outputs they are mutually near-orthogonal by construction; the interesting signal is their correlation with `Class`.

In [ ]:
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True,
            cbar_kws={'shrink': 0.6}, ax=ax)
ax.set_title('Feature correlation matrix')
plt.tight_layout(); plt.show()

In [ ]:
# Rank features by absolute correlation with the target.
target_corr = corr['Class'].drop('Class').abs().sort_values(ascending=False)
top = target_corr.head(10)
print('Top 10 features by |corr| with Class:')
print(top.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=top.values, y=top.index, ax=ax, color='#c44e52')
ax.set_title('|Pearson correlation with Class|'); ax.set_xlabel('|corr|')
plt.tight_layout(); plt.show()

## 4. Distribution of the most discriminative features

Overlaid class-conditional densities for the top-correlated PCA components.

In [ ]:
top_feats = top.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for feat, ax in zip(top_feats, axes.ravel()):
    sns.kdeplot(legit[feat], ax=ax, label='legit', fill=True, alpha=0.3)
    sns.kdeplot(fraud[feat], ax=ax, label='fraud', fill=True, alpha=0.3)
    ax.set_title(feat); ax.legend()
plt.tight_layout(); plt.show()

## 5. Statistical hypothesis testing

For each feature we test whether the fraud and legit populations differ.

- **Mann–Whitney U** (non-parametric): does feature X have a different distribution across classes? The PCA components are not Gaussian, so we prefer this over a t-test.
- **Welch's t-test**: reported alongside for reference (unequal variances).
- **Effect size**: rank-biserial correlation from the U statistic, so a tiny p-value on 284K rows isn't mistaken for a large effect.

H0: the two classes share the same distribution for feature X.

In [ ]:
def rank_biserial(u, n1, n2):
    """Effect size for Mann-Whitney U in [-1, 1]."""
    return 1 - (2 * u) / (n1 * n2)

features = [c for c in df.columns if c != 'Class']
rows = []
n1, n2 = len(fraud), len(legit)
for feat in features:
    u, p_mw = stats.mannwhitneyu(fraud[feat], legit[feat], alternative='two-sided')
    t, p_t = stats.ttest_ind(fraud[feat], legit[feat], equal_var=False)
    rows.append({
        'feature': feat,
        'mannwhitney_p': p_mw,
        'welch_t_p': p_t,
        'effect_size': rank_biserial(u, n1, n2),
    })

results = pd.DataFrame(rows)
# Bonferroni correction across the ~30 simultaneous tests.
alpha = 0.05
results['bonferroni_sig'] = results['mannwhitney_p'] < (alpha / len(features))
results = results.sort_values('effect_size', key=np.abs, ascending=False)
results.reset_index(drop=True).head(15)

In [ ]:
n_sig = int(results['bonferroni_sig'].sum())
print(f'{n_sig}/{len(features)} features differ significantly between classes '
      f'(Mann-Whitney, Bonferroni alpha={alpha}).')
strong = results[results.effect_size.abs() > 0.3]['feature'].tolist()
print('Features with |effect size| > 0.3 (practically meaningful):')
print(strong)

### Takeaways for modeling

1. **Extreme imbalance (0.172%)** → accuracy is useless; optimize for recall/PR-AUC and resample (SMOTE) on the training fold only.
2. A handful of PCA components (e.g. `V14`, `V17`, `V12`, `V10`, `V16`) carry most of the class separation and show large effect sizes, not just small p-values.
3. `Amount` and `Time` are on a different scale from the PCA features → **RobustScaler** (outlier-tolerant) before modeling.
4. The strong, mostly-linear signal in several features is why even Logistic Regression is a competitive baseline — the ANN's job is to squeeze out the remaining non-linear interactions and lift recall.

These decisions are implemented in `src/data_preprocessing.py` and `src/train.py`.